# 06 — Filtering predictions by uncertainty (27-class RBP)

The practical payoff: keep only the predictions whose uncertainty score for the
predicted class is below 0.1, and average precision rises (Fig. 12). The supplement
adds the full coverage sweep with bootstrap intervals (Fig. S7, `APP27_ap_coverage`).

This notebook needs the raw per-iteration BNN predictions rather than the summary
bundle. The threshold is defined in probability space and the softmax is
non-linear, so the standard deviation has to be computed after applying the softmax
to each Monte-Carlo pass; it cannot be recovered from a stored SD of logits.

In [ ]:
import os, sys, pathlib

# repo root, whether the notebook is run from notebooks/ or from the root
ROOT = pathlib.Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))

# released data: the summary bundle (all notebooks) and the raw per-pass predictions
# (notebooks 04 and 06). Set the variables before starting Jupyter to use other locations.
os.environ.setdefault('UQFIG_BUNDLE', str(ROOT / 'figure_data_bundle'))
os.environ.setdefault('UQFIG_PREDS', str(ROOT / 'cluster_preds'))
OUT = ROOT / 'out'; OUT.mkdir(exist_ok=True)

from uq_genomics.figures import style as S
S.apply_rc()
print('bundle :', os.environ['UQFIG_BUNDLE'])
print('preds  :', os.environ['UQFIG_PREDS'])
print('output :', OUT)

In [ ]:
PRED = pathlib.Path(os.environ['UQFIG_PREDS']) / 'iter_preds_bnn_v_10_s_9.pickle'
print(PRED, '->', 'found' if PRED.exists() else 'MISSING - see docs/REPRODUCING.md')

## Fig. 12 — per-class average precision before and after filtering

In [ ]:
from uq_genomics.figures import plots_rbp27 as CF
CF.build_app27(str(OUT), n_iter=100, extended=True)

In [ ]:
from IPython.display import IFrame, display
p = OUT / 'APP27_filtering_extended.pdf'
print(p, '->', 'exists' if p.exists() else 'MISSING')
display(IFrame(str(p), width=900, height=560)) if p.exists() else None

## Coverage sweep (Fig. S7)

In [ ]:
from uq_genomics.figures import plots_selective as PSEL
d, summary, _ = PSEL.build(str(OUT))

In [ ]:
from IPython.display import IFrame, display
p = OUT / 'APP27_ap_coverage.pdf'
print(p, '->', 'exists' if p.exists() else 'MISSING')
display(IFrame(str(p), width=900, height=420)) if p.exists() else None

## Why a few classes show no gain

Average precision has a floor equal to the class prevalence. Filtering removes
proportionally more positives than negatives for the classes the model is unsure
about, so their prevalence falls and the floor falls with it. Measured against that
floor most of them still improve.

In [ ]:
import numpy as np
from sklearn.metrics import average_precision_score
from uq_genomics.figures import data_access as DA
raw = CF._load_iters(str(PRED))[:100]
pr = CF._softmax(raw); mean, sd = pr.mean(0), pr.std(0)
y = np.asarray(DA.bundle_part('app27')['y_test'])
keep = sd[np.arange(len(sd)), mean.argmax(1)] <= 0.1
print(f'{"class":10s}{"AP0":>8s}{"AP1":>8s}{"dAP":>8s}{"lift0":>8s}{"lift1":>8s}')
for c in range(y.shape[1]):
    p0, p1 = y[:, c].mean(), y[keep][:, c].mean()
    a0 = average_precision_score(y[:, c], mean[:, c])
    a1 = average_precision_score(y[keep][:, c], mean[keep][:, c])
    if a1 - a0 < 0:
        print(f'{CF.RBP27_NAMES[c]:10s}{a0:>8.3f}{a1:>8.3f}{a1-a0:>+8.3f}{a0/p0:>8.2f}{a1/p1:>8.2f}')